<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-15_March-19-2026/PCA_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Initially generated by Claude

#@markdown # Principal Component Analysis (PCA) Example
#@markdown This uses numpy to do the PCA.


import numpy as np
import matplotlib.pyplot as plt

#@markdown ### Number of Samples
Number_of_Samples = 2000 #@param

#@markdown ### Angle
Angle_Degrees = 35 #@param

#@markdown ### Spread
#@markdown #### - Dimension 1
Spread_Dim1 = 2.0 #@param
#@markdown #### - Dimension 2
Spread_Dim2 = 0.4 #@param

# --- 1. Generate correlated 2D data ---
rng = np.random.default_rng(42)
n = Number_of_Samples
angle = np.radians(Angle_Degrees)

# Data aligned to a rotated axis: large spread along one direction, small along the other
spread = np.array([Spread_Dim1, Spread_Dim2])
latent = rng.standard_normal((n, 2)) * spread          # uncorrelated

rotation = np.array([
    [np.cos(angle), -np.sin(angle)],
    [np.sin(angle),  np.cos(angle)]
])
X = latent @ rotation.T                                 # now correlated

# --- 2. Center the data ---
X_mean = X.mean(axis=0)
X_centered = X - X_mean

# --- 3. Compute the covariance matrix ---
cov = (X_centered.T @ X_centered) / (n - 1)
print("Covariance matrix:\n", np.round(cov, 3))

# --- 4. Eigen-decomposition ---
eigenvalues, eigenvectors = np.linalg.eigh(cov)

# eigh returns eigenvalues in ascending order — reverse to get PC1 first
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]   # columns are principal components

pc1, pc2 = eigenvectors[:, 0], eigenvectors[:, 1]

# --- 5. Variance explained ---
total_var = eigenvalues.sum()
explained = eigenvalues / total_var * 100
print(f"\nPC1 explains {explained[0]:.1f}% of variance")
print(f"PC2 explains {explained[1]:.1f}% of variance")

# --- 6. Project data onto principal components ---
X_pca = X_centered @ eigenvectors      # shape (n, 2)

# --- 7. Reconstruct using only PC1 (dimensionality reduction) ---
X_reconstructed = X_pca[:, :1] @ eigenvectors[:, :1].T + X_mean

# --- 8. Plot ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

scale = 2.5  # arrow length in data units

ax = axes[0]
ax.scatter(*X.T, alpha=0.5, s=25, color='steelblue', label='Data')
for vec, val, color, name in zip(
    eigenvectors.T, eigenvalues,
    ['tomato', 'seagreen'], ['PC1', 'PC2']
):
    ax.annotate("", xy=X_mean + vec * val**0.5 * scale,
                xytext=X_mean - vec * val**0.5 * scale,
                arrowprops=dict(arrowstyle="<->", color=color, lw=2))
    ax.text(*(X_mean + vec * val**0.5 * scale * 1.1), name,
            color=color, fontsize=10, ha='center')
ax.set_aspect('equal')
ax.set_title('Original data + principal components')
ax.legend(loc='upper left', fontsize=9)

ax = axes[1]
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=25, color='steelblue')
ax.axhline(0, color='tomato', lw=1.5, label=f'PC1 ({explained[0]:.1f}%)')
ax.axvline(0, color='seagreen', lw=1.5, linestyle='--', label=f'PC2 ({explained[1]:.1f}%)')
ax.set_xlabel('PC1 score')
ax.set_ylabel('PC2 score')
ax.set_title('Projected (PCA space)')
ax.legend(fontsize=9)

ax = axes[2]
ax.scatter(*X.T, alpha=0.3, s=25, color='steelblue', label='Original')
ax.scatter(*X_reconstructed.T, alpha=0.6, s=25, color='tomato', label='PC1 only (reconstruction)')
ax.set_aspect('equal')
ax.set_title('Reconstruction from PC1 alone')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Initially generated by Claude

#@markdown # Principal Component Analysis (PCA) Example
#@markdown This uses scikit-learn to do the PCA.


import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


#@markdown ### Number of Samples
Number_of_Samples = 2000 #@param

#@markdown ### Angle
Angle_Degrees = 35 #@param

#@markdown ### Spread
#@markdown #### - Dimension 1
Spread_Dim1 = 2.0 #@param
#@markdown #### - Dimension 2
Spread_Dim2 = 0.4 #@param

# --- 1. Generate correlated 2D data ---
rng = np.random.default_rng(42)
n = Number_of_Samples
angle = np.radians(Angle_Degrees)

# Data aligned to a rotated axis: large spread along one direction, small along the other
spread = np.array([Spread_Dim1, Spread_Dim2])
latent = rng.standard_normal((n, 2)) * spread          # uncorrelated

rotation = np.array([
    [np.cos(angle), -np.sin(angle)],
    [np.sin(angle),  np.cos(angle)]
])
X = latent @ rotation.T                                 # now correlated

pca = PCA()
pca.fit(X)

eigenvectors = pca.components_
eigenvalues = pca.explained_variance_

# --- 5. Variance explained ---
explained = pca.explained_variance_ratio_ * 100
print(f"\nPC1 explains {explained[0]:.1f}% of variance")
print(f"PC2 explains {explained[1]:.1f}% of variance")

# --- 6. Project data onto principal components ---
X_pca = pca.transform(X)

# --- 7. Reconstruct using only PC1 (dimensionality reduction) ---
X_reconstructed = pca.inverse_transform(
    np.column_stack([X_pca[:, 0], np.zeros(n)])
)

# --- 8. Plot ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

scale = 2.5  # arrow length in data units

ax = axes[0]
ax.scatter(*X.T, alpha=0.5, s=25, color='steelblue', label='Data')
for i, (color, name) in enumerate(zip(['tomato', 'seagreen'], ['PC1', 'PC2'])):
    vec = eigenvectors[i]          # sklearn: rows are components
    val = eigenvalues[i]
    ax.annotate("", xy=X_mean + vec * val**0.5 * scale,
                xytext=X_mean - vec * val**0.5 * scale,
                arrowprops=dict(arrowstyle="<->", color=color, lw=2))
    ax.text(*(X_mean + vec * val**0.5 * scale * 1.1), name,
            color=color, fontsize=10, ha='center')
ax.set_aspect('equal')
ax.set_title('Original data + principal components')
ax.legend(loc='upper left', fontsize=9)

ax = axes[1]
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=25, color='steelblue')
ax.axhline(0, color='tomato', lw=1.5, label=f'PC1 ({explained[0]:.1f}%)')
ax.axvline(0, color='seagreen', lw=1.5, linestyle='--', label=f'PC2 ({explained[1]:.1f}%)')
ax.set_xlabel('PC1 score')
ax.set_ylabel('PC2 score')
ax.set_title('Projected (PCA space)')
ax.legend(fontsize=9)

ax = axes[2]
ax.scatter(*X.T, alpha=0.3, s=25, color='steelblue', label='Original')
ax.scatter(*X_reconstructed.T, alpha=0.6, s=25, color='tomato', label='PC1 only (reconstruction)')
ax.set_aspect('equal')
ax.set_title('Reconstruction from PC1 alone')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()